# 01 · Online AMT: 피아노 오디오 → MIDI (실시간)

**실시간 피아노 채보** (Kwon, Jeong & Nam. ISMIR 2020 LBD · PAR 모델의 전신). CNN이 mel-spectrogram에서 특징을 뽑고, 건반 88개마다 autoregressive LSTM이 5가지 상태(off · offset · sustain · onset · re-onset)를 매 32 ms 프레임마다 결정합니다. 앞 프레임의 예측을 다음 입력으로 쓰므로 **한 방향으로만** 처리할 수 있고, 그래서 실시간이 됩니다.

- 입력 → 출력: 피아노 오디오 → 프레임별 note 상태 → MIDI
- 소요 시간: 설치 1분 + 30초 곡 채보 CPU 약 10초
- GPU 불필요 (CPU 1스레드로도 실시간의 2배 속도)
- 논문: Kwon, Jeong & Nam, "Towards Efficient and Real-Time Piano Transcription Using Neural Autoregressive Models," IEEE/ACM TASLP 2024 · 코드: https://github.com/jdasam/online_amt

In [ ]:
#@title 0. 준비: 수업 자료 받기와 모델 폴더 정하기 (실행만 하면 됩니다)
#@markdown 모델 가중치는 모두 `models` 폴더 하나에 받습니다. 기본 위치는 `/content`이고 런타임을 끄면 지워집니다. 체크하면 Google Drive의 `CT-AI_Microdegree/` 아래에 `models`와 `outputs`를 두어 다음 실행 때 다시 받지 않고, 다른 노트북의 결과도 이어서 쓸 수 있습니다.
KEEP_IN_DRIVE = False  #@param {type:"boolean"}
import os, sys, glob, time, json, subprocess
_cands = [os.environ.get('CTAI_REPO_DIR'), os.path.abspath(os.path.join(os.getcwd(), '..', '..')), '/content/CT-AI_Microdegree']
REPO = next((c for c in _cands if c and os.path.isfile(os.path.join(c, 'colab', 'ctai.py'))), None)
if REPO is None:
    REPO = '/content/CT-AI_Microdegree'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/dabinkim0/CT-AI_Microdegree', REPO], check=True)
sys.path.insert(0, os.path.join(REPO, 'colab'))
REQ = os.path.join(REPO, 'colab', 'requirements')   # 노트북별 pip 패키지 목록
import ctai
P = ctai.setup(keep_in_drive=KEEP_IN_DRIVE)
AUDIO_DIR, ASSET_DIR, MODELS_DIR, CODE_DIR, OUT_DIR, IN_COLAB = P['AUDIO_DIR'], P['ASSET_DIR'], P['MODELS_DIR'], P['CODE_DIR'], P['OUT_DIR'], P['IN_COLAB']
print('예시 음원:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(AUDIO_DIR, '*.wav'))))


In [ ]:
#@title 설치
%pip install -q -r {REQ}/01_online_amt.txt
import subprocess, sys, os
AMT_DIR = os.path.join(CODE_DIR, 'online_amt')                     # 코드 (보통 모델 폴더 안)
AMT_COMMIT = 'ad12550909a1d86f699097d11885f427054a5ac2'           # 저자 저장소의 커밋을 고정
ctai.git_checkout('https://github.com/jdasam/online_amt', AMT_DIR, AMT_COMMIT)
# 최신 librosa에 맞춘 2줄 패치
p = os.path.join(AMT_DIR, 'autoregressive', 'mel.py'); s = open(p).read()
s = s.replace('pad_center(fft_window, filter_length)', 'pad_center(fft_window, size=filter_length)')
s = s.replace('mel(sample_rate, filter_length, n_mels, mel_fmin, mel_fmax, htk=True)', 'mel(sr=sample_rate, n_fft=filter_length, n_mels=n_mels, fmin=mel_fmin, fmax=mel_fmax, htk=True)')
open(p, 'w').write(s)
import torch, librosa; print('설치 OK · torch', torch.__version__, '· librosa', librosa.__version__)

In [ ]:
#@title 오디오 준비: 예시 음원 / 업로드 / YouTube 중 선택
import ipywidgets as W, soundfile as sf, numpy as np, librosa
from IPython.display import Audio, display
import matplotlib.pyplot as plt

EXAMPLES = {'피아노 (Scarlatti K.455, MAESTRO)': 'piano_scarlatti_k455_30s.wav'}
SOURCE = '예시'      #@param ["예시", "업로드", "YouTube"]
EXAMPLE = '피아노 (Scarlatti K.455, MAESTRO)'   #@param ["피아노 (Scarlatti K.455, MAESTRO)"]
YOUTUBE_URL = ''   #@param {type:"string"}
MAX_SECONDS = 30   #@param {type:"integer"}
TARGET_SR = 16000

def load_audio(path, sr=TARGET_SR, max_sec=MAX_SECONDS, mono=True):
    y, _ = librosa.load(path, sr=sr, mono=mono)
    if max_sec: y = y[..., :int(max_sec * sr)]
    return y, sr

if SOURCE == '예시':
    audio_path = os.path.join(AUDIO_DIR, EXAMPLES[EXAMPLE])
elif SOURCE == '업로드':
    from google.colab import files
    up = files.upload(); audio_path = list(up.keys())[0]
else:
    # 수업 내 학습 목적으로만 사용하세요. 저작권이 있는 음원은 외부에 배포하지 않습니다.
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'], check=True)
    subprocess.run(['yt-dlp', '-q', '-x', '--audio-format', 'wav', '-o', 'yt_input.%(ext)s', YOUTUBE_URL], check=True)
    audio_path = 'yt_input.wav'

y, sr = load_audio(audio_path)
mono = y if y.ndim == 1 else y.mean(0)
print('파일:', audio_path, '| 길이: %.1f s | sr=%d' % (len(mono) / sr, sr))
plt.figure(figsize=(14, 2)); plt.plot(np.arange(len(mono)) / sr, mono, lw=.4, color='#58c4dd'); plt.xlabel('time (s)'); plt.title('waveform'); plt.tight_layout(); plt.show()
display(Audio(mono, rate=sr))


In [ ]:
#@title 모델 로드 (체크포인트 179 MB, 처음 한 번만 내려받음)
import torch
ckpt = str(ctai.fetch(['https://media.githubusercontent.com/media/jdasam/online_amt/%s/model-180000.pt' % AMT_COMMIT,
                       'gdrive:12DnYJJ6YKpsoEkXI9fYUTTd4wOeY_rlB'],                  # 저자가 README에 올린 Google Drive 사본
                      os.path.join(MODELS_DIR, 'online_amt', 'model-180000.pt'), size=178804960))
sys.path.insert(0, AMT_DIR); sys.path.insert(0, ASSET_DIR)
from transcribe import load_model, OnlineTranscriber
from run_piano_amt import RawStateTranscriber, run_online, states_to_notes, velocity_proxy_from_audio, FPS
model = load_model(ckpt)
print('파라미터 수: %.1f M' % (sum(p.numel() for p in model.parameters()) / 1e6), '| 프레임률:', FPS, 'fps (hop 32 ms)')

In [ ]:
#@title 추론: 32 ms 청크로 실시간 루프 시뮬레이션
torch.set_num_threads(2)
t0 = time.time()
states, _total, step_s = run_online(model, mono)   # (frames, 88) 상태 id, 총 시간, 청크당 처리 시간(초)
dt = time.time() - t0; dur = len(mono) / sr
DELAY = 0.16   # 모델 구조상 160 ms (7프레임 버퍼 중앙) + 청크 32 ms + 연산 ≈ 200 ms
print('처리 %.1f s / 음원 %.1f s → 실시간의 %.1f배 | 청크당 %.1f ms (청크 길이 32 ms)' % (dt, dur, dur / dt, float(np.mean(step_s) * 1000)))
vel = velocity_proxy_from_audio(mono, len(states))
notes_raw = states_to_notes(states, FPS, velocity_proxy=vel, delay_sec=0.0)      # 보정 없음: 소리보다 0.2초 늦음
notes = states_to_notes(states, FPS, velocity_proxy=vel, delay_sec=DELAY)        # 지연 보정 (오프라인 분석용)
print('note 수:', len(notes))

In [ ]:
#@title 정답과 비교 (예시 음원은 연주자의 MIDI가 있음)
import pretty_midi, mir_eval
gt_path = os.path.join(AUDIO_DIR, 'piano_scarlatti_k455_30s.mid')
gt = [(n.start, n.end, n.pitch) for i in pretty_midi.PrettyMIDI(gt_path).instruments for n in i.notes] if (SOURCE == '예시' and os.path.exists(gt_path)) else []
def f1(nts, ref):
    if not ref or not nts: return None
    a = np.array([[n['onset'], n['offset']] for n in nts]); p = np.array([librosa.midi_to_hz(n['pitch']) for n in nts])
    ra = np.array([[s, e] for s, e, _ in ref]); rp = np.array([librosa.midi_to_hz(m) for _, _, m in ref])
    return mir_eval.transcription.precision_recall_f1_overlap(ra, rp, a, p, offset_ratio=None)[2]
fa, fb = f1(notes_raw, gt), f1(notes, gt)
print('onset F1 (50 ms 허용) · 지연 보정 전: %s · 보정 후: %s' % ('%.2f' % fa if fa is not None else '정답 없음', '%.2f' % fb if fb is not None else '정답 없음'))

In [ ]:
#@title 합성 도우미: note 목록을 사인파로 바꿔 듣기 (데모 페이지의 합성과 같은 방식)
import numpy as np
def midi_to_hz(m): return 440.0 * 2 ** ((np.asarray(m, dtype=float) - 69) / 12)
def synth_notes(notes, sr=22050, duration=None, gain=0.12, drums=False):
    """notes: [(onset, offset, pitch), ...] → float32 waveform. drums=True면 노이즈 버스트."""
    if duration is None: duration = max(n[1] for n in notes) + 0.5 if notes else 1.0
    out = np.zeros(int(duration * sr) + 1, dtype=np.float32)
    rng = np.random.default_rng(0)
    for on, off, pitch in notes:
        a = int(on * sr); b = int(max(off, on + 0.08) * sr) + int(0.05 * sr)
        if a >= len(out): continue
        n = min(b, len(out)) - a; t = np.arange(n) / sr
        if drums:
            kick = pitch in (35, 36); dec = 0.18 if kick else 0.08
            env = np.exp(-t / dec); sig = rng.standard_normal(n) * 0.5
            if kick: sig = sig * 0.3 + np.sin(2 * np.pi * (150 * np.exp(-t / 0.1) + 45) * t) * 0.9
            out[a:a + n] += (sig * env * gain * 3).astype(np.float32)
        else:
            env = np.minimum(1, t / 0.008) * np.where(t < 0.25, 1 - 0.65 * t / 0.25, 0.35)
            env[-int(0.03 * sr):] *= np.linspace(1, 0, int(0.03 * sr))
            out[a:a + n] += (np.sin(2 * np.pi * midi_to_hz(pitch) * t) * env * gain).astype(np.float32)
    peak = np.abs(out).max(); return out / peak * 0.8 if peak > 0 else out
def mix(*tracks):
    n = max(len(t) for t in tracks); s = np.zeros(n, np.float32)
    for t in tracks: s[:len(t)] += t
    peak = np.abs(s).max(); return s / peak * 0.8 if peak > 0 else s


In [ ]:
#@title 듣기·보기: 피아노롤 + 사인파 합성
fig, ax = plt.subplots(figsize=(14, 4.5))
for s_, e_, p_ in gt: ax.plot([s_, e_], [p_, p_], color='#f5d547', lw=4, alpha=.5, solid_capstyle='butt')
for n in notes: ax.plot([n['onset'], n['offset']], [n['pitch'], n['pitch']], color='#58c4dd', lw=2, solid_capstyle='butt')
ax.set_xlabel('time (s)'); ax.set_ylabel('MIDI pitch'); ax.set_title('blue: model output (delay-compensated) · yellow: ground truth'); plt.tight_layout(); plt.show()
synth = synth_notes([(n['onset'], n['offset'], n['pitch']) for n in notes], sr=sr, duration=dur)
print('원음'); display(Audio(mono, rate=sr))
print('채보 결과 (사인파)'); display(Audio(synth, rate=sr))
print('원음(왼쪽) + 채보(오른쪽)'); display(Audio(np.stack([mono[:len(synth)], synth[:len(mono)]]), rate=sr))

## 해 보기

1. **지연을 귀로 듣기**: 아래 셀은 지연 보정을 뺀 결과(`notes_raw`)를 원음과 겹쳐 재생합니다. 0.2초 늦게 따라오는 것이 들리나요? 실시간 비주얼에서는 이 지연을 어떻게 다루면 좋을까요?
2. **velocity와 offset**: 이 모델은 세기(velocity)를 출력하지 않아 음량으로 추정하고, 페달을 많이 쓰는 곡에서는 offset이 흔들립니다. 자기 피아노 음원을 올려 보고(`SOURCE='업로드'`), 어떤 음이 빠지거나 늘어지는지 찾아 보세요.

In [ ]:
#@title 지연 보정 전/후 비교 재생
synth_raw = synth_notes([(n['onset'], n['offset'], n['pitch']) for n in notes_raw], sr=sr, duration=dur)
print('보정 전 (실시간 그대로, 0.2 s 늦음): 원음 L / 합성 R'); display(Audio(np.stack([mono[:len(synth_raw)], synth_raw[:len(mono)]]), rate=sr))
print('보정 후: 원음 L / 합성 R'); display(Audio(np.stack([mono[:len(synth)], synth[:len(mono)]]), rate=sr))

In [ ]:
#@title 저장: MIDI와 note JSON
import pretty_midi
pm = pretty_midi.PrettyMIDI(); inst = pretty_midi.Instrument(0)
for n in notes: inst.notes.append(pretty_midi.Note(int(n.get('velocity', 80)), int(n['pitch']), float(n['onset']), float(n['offset'])))
pm.instruments.append(inst); midi_out = os.path.join(OUT_DIR, 'online_amt.mid'); pm.write(midi_out)
json.dump(notes, open(os.path.join(OUT_DIR, 'online_amt_notes.json'), 'w')); print('저장:', midi_out)